## 토픽 모델링 실제

In [ ]:
# 폴더구성
# TopicModeling_1/dataset/1.Data/contents.xlsx : 소스문서파일(뉴스), Data_stop_word : 불용어 파일
# TopicModeling_1/dataset/2.Model/ : 실험 중
# TopicModeling_1/dataset/3.Result/: 실험 결과

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
#파일 업로드
import shutil
try:
  shutil.copy('/content/gdrive/MyDrive/text/11주차/토픽2/dataset.zip', '/content/')
  print('dataset download!!!')
except Exception as err:
  print(str(err))

In [ ]:
# 압축 풀고 디렉토리 생성

import zipfile

with zipfile.ZipFile('/content/dataset.zip', 'r') as target_file:
    target_file.extractall('/content/dataset/')
print('ok')

### Package Import

In [ ]:
!pip install ujson
!pip install gensim

In [ ]:
!pip install konlpy #한국어 정보처리
!pip install pyLDAvis #토픽모델링의 LDA 모델의 학습결과를 시각적으로 표현하는 라이브러리

In [ ]:
# Data Preprocessing Package
import re                       #정규식
import numpy as np
import pandas as pd
import os                      #디렉토리와 경로정보

# NLP Package
from konlpy.tag import * 
import gensim                    #토픽모델링을 하는 라이브러리
import gensim.corpora as corpora #텍스트분석
from gensim.models import CoherenceModel
from collections import Counter

# Visualization Package   #LDA시각화
import pyLDAvis 
import pyLDAvis.gensim_models 
import matplotlib.pyplot as plt
%matplotlib inline


from pprint import pprint #pretty print
import itertools #iterable 객체 처리
import math

import logging #로그처리
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.ERROR)
 
import warnings #경고 무시
warnings.filterwarnings("ignore",category=DeprecationWarning)

### 데이터 경로 및 저장 경로 설정

In [ ]:
# 기본 저장 주소
ROOT_PATH ="./dataset/"
# 데이터 저장 주소
DATA_FOLDER_PATH = os.path.join(ROOT_PATH,"1.Data/")
DATA_FILE_NAME = os.path.join(DATA_FOLDER_PATH,"contents.xlsx")                  # Portal News Crawler에서 저장한 데이터 
DATA_STOP_WORD_FILE_NAME = os.path.join(DATA_FOLDER_PATH,"Data_stop_word.txt")   # 불용어 사전 

# 모델 저장 주소
MODEL_SAVE_FOLDER = os.path.join(ROOT_PATH,"2.Model/")
LDA_MODEL_SAVE_NAME = os.path.join(MODEL_SAVE_FOLDER, "Model_lda_topic_modeling.lda")
OPT_MODEL_SAVE_NAME = os.path.join(MODEL_SAVE_FOLDER, "Model_opt_topic_modeling.lda")

# 결과물 저장 주소
RESULT_FOLDER = os.path.join(ROOT_PATH,"3.Result/")
RESULT_SAVE_LDAVIS = os.path.join(RESULT_FOLDER,"Result_lda_vis.html")
RESULT_TOPIC_EXCEL = os.path.join(RESULT_FOLDER,"Result_topic_excel.xlsx")
RESULT_corpus_EXCEL = os.path.join(RESULT_FOLDER,"Result_corpus_excel.xlsx")

### Data Loading

- Portal_News_Crawler에서 수집한 데이터에서 Text 정보만 불러옴

In [ ]:
# 데이터 불러오기 
DF_raw = pd.read_excel(DATA_FILE_NAME,sheet_name='sheet1') # 엑셀파일의 sheet1시트의 텍스트 가져오기
len(DF_raw)

# 불러온 데이터의 값이 비어 있는지 확인
print('Null값이 있는지 확인합니다.',DF_raw.isnull().values.any()) # Null 값이 존재하는지 확인 (False=정상)
print('')
DF_raw = DF_raw.dropna(how = 'any') # Null 값이 존재하는 행 제거
DF_raw = DF_raw.drop_duplicates()   # 중복 데이터 프레임 제거 
DF_raw = DF_raw.reset_index(drop=True) # 데이터 프레임 재색인
print('중복 및, NULL값을 제거한 후, 다시 NULL값을 확인 합니다.', DF_raw.isnull().values.any()) # Null 값이 존재하는지 확인 (False=정상)
print('')
print("처리할 데이터수 : ",len(DF_raw))

# raw데이터로부터 텍스트만 불러오기 
DF_only_text = DF_raw['text'] #엑셀필드명

In [ ]:
df_jl = pd.read_csv('jeongleungfood.csv')

In [ ]:
df_jl.content

### 토크나이징, 불용어 처리, 말뭉치 생성, 빈도 계수

In [ ]:
tokenizer = Okt() # 토큰나이저 지정
stopword_vocab =['정릉','시장','맛집'] # 불용어 파일 불러오기
sep = "\n" # 불용어 처리 인자

def build_vocab(data_frame ,stopword_vocab, separate):
    
    # 불용어 데이터를 가져와 리스트로 변환합니다.
    # with open(stopword_vocab, encoding = 'utf-8') as f:
    #    temp1 = []
    #    for i in f:
    #        temp1.append(i)
    #        
    # globals()['stopword_vocab'] = []
    
    # 불용어 데이터는 전역변수 stopword_vocab 선언합니다. 
    # 구분자에 따라 stopword_vocab에 추가하여 불용어 사전을 구축합니다.
    # for j in range(len(temp1)):
    #   temp2 = temp1[j].rstrip(separate)
    #    globals()['stopword_vocab'].append(temp2)
    
    #okt token에서 명사만 출력합니다. 단어의 길이가 1 초과인 단어만 출력합니다. 
    globals()['list_sent2words'] =[]
    for i in range(len(data_frame)) :
        num_list=[]
        temp = tokenizer.nouns(data_frame[i])
        for j in range(len(temp)):

            if len(temp[j]) > 1:
                num_list.append(temp[j])
        globals()['list_sent2words'].append(num_list)
    
    return [[word for word in doc if word not in globals()['stopword_vocab']] for doc in globals()['list_sent2words']]

result_data =build_vocab(df_jl.content, stopword_vocab, sep)


# 전체 에 대한 워드 카운트 계수 확인

def word_corpus(result_data):
    #전체 단어의 갯수 파악
    words = list(itertools.chain(*result_data))
    print('전체 워드의 개수 : {}'.format(len(words)))

    #단어의 빈도수를 확인 후 추가할 불용어 확인 작업
    vocab = Counter(words)
    vocab_size = len(words)
    vocab = vocab.most_common(vocab_size) # 등장 빈도수가 높은 상위 n개의 단어만 저장 vocab
    return vocab

vocab=word_corpus(result_data)

# 전체 워드의 빈도 계수 
df_corpus=pd.DataFrame(columns=["text","count"])
tmp_list=[]
tmp_list1=[]
for word, num in vocab:
    tmp_list.append(word)
    tmp_list1.append(num)
df_corpus['text']=tmp_list
df_corpus['count']=tmp_list1
#상위 20개의 워드 카운드 계수만 출력
a=df_corpus.head(1000)
print(df_corpus.head(20))

#빈도분석한 결과를 별도의 파일에 저장함
a.to_excel(RESULT_corpus_EXCEL ,sheet_name = "sheet1")


# 토픽 모델링 딕셔너리 생성
id2word = corpora.Dictionary(result_data)
 
# 토픽모델링에 사용할 말뭉치 생성
texts = result_data
 
# 용어-문서 빈도
corpus = [id2word.doc2bow(text) for text in texts]


### 토픽 평가

In [ ]:
# Perplexity와 Coherence Score 을 판단
#print('토픽 기본 모델링을 실시 합니다. 해당 모델은 "lda_model" 변수로 입력됩니다.')
#print(' ')

#NUM_TOPICS = int(input('토픽의 개수를 입력해 주세요. '))
#TOPICS_W_NUM = int(input('출력할 토픽별 단어의 개수를 입력해 주세요 '))
#save_lda_model= int(input("선택한 토픽 모델을 저장하시겠습니까? \n0 저장  \n1 미저장  "))

TOPICS_W_NUM =20
save_lda_model=0
RANDOM_STATE = 100
UPDATE_EVERY = 1
CHUNKSIZE = 100
PASSES = 10
ALPHA = 'auto'
PER_WORD_TOPICS = True
print('NUM_TOPICS', 'perplexity', 'coherence')
for i in range(1,30):
  NUM_TOPICS=i
 
  #해당 셀은 토픽모델링(LDA)에 대해 모델을 정의하는 셀입니다.
  lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus, id2word=id2word, 
                                              num_topics=NUM_TOPICS, random_state=RANDOM_STATE, 
                                              update_every=UPDATE_EVERY, chunksize=CHUNKSIZE,
                                              passes=PASSES, alpha=ALPHA, per_word_topics=PER_WORD_TOPICS)

  # 토픽 출력
 #  pprint(lda_model.print_topics(num_words=TOPICS_W_NUM))
  doc_lda = lda_model[corpus]

  # 모델 저장 
  if save_lda_model == 0:
      lda_model.save(LDA_MODEL_SAVE_NAME)
  # 0번 토픽,- 중요단어들이 가중치 순으로 나옴(20개)
  """
  해당 셀은 설계한 모델을 계산하는 셀입니다.
  측정은 Perplexity와 Coherence Score입니다.
  """
  #print('\nNUM_TOPICS',NUM_TOPICS)
  # Perplexity 
  #print('Perplexity: ', lda_model.log_perplexity(corpus)) # a measure of how good the model is. lower the better.
  
  # Coherence Score
  coherence_model_lda = CoherenceModel(model=lda_model, texts=result_data, dictionary=id2word, coherence='c_v')
  coherence_lda = coherence_model_lda.get_coherence()
  #print('Coherence Score: ', coherence_lda)
 # print('NUM_TOPICS',NUM_TOPICS,'Perplexity: ', lda_model.log_perplexity(corpus),'Coherence: ', coherence_lda)

  print('T',NUM_TOPICS, lda_model.log_perplexity(corpus), coherence_lda)


In [ ]:
print('토픽 기본 모델링을 실시 합니다. 해당 모델은 "lda_model" 변수로 입력됩니다.')
print(' ')

NUM_TOPICS = int(input('토픽의 개수를 입력해 주세요. '))
TOPICS_W_NUM = int(input('출력할 토픽별 단어의 개수를 입력해 주세요 '))
save_lda_model= int(input("선택한 토픽 모델을 저장하시겠습니까? \n0 저장  \n1 미저장  "))

RANDOM_STATE = 100
UPDATE_EVERY = 1
CHUNKSIZE = 100
PASSES = 10
ALPHA = 'auto'
PER_WORD_TOPICS = True

#해당 셀은 토픽모델링(LDA)에 대해 모델을 정의하는 셀입니다.
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus, id2word=id2word, 
                                            num_topics=NUM_TOPICS, random_state=RANDOM_STATE, 
                                            update_every=UPDATE_EVERY, chunksize=CHUNKSIZE,
                                            passes=PASSES, alpha=ALPHA, per_word_topics=PER_WORD_TOPICS)

# 토픽 출력
pprint(lda_model.print_topics(num_words=TOPICS_W_NUM))
doc_lda = lda_model[corpus]

# 모델 저장 
if save_lda_model == 0:
    lda_model.save(LDA_MODEL_SAVE_NAME)
# 0번 토픽,- 중요단어들이 가중치 순으로 나옴(20개)

In [ ]:
#토픽평가
"""
해당 셀은 설계한 모델을 계산하는 셀입니다.
측정은 Perplexity와 Coherence Score입니다.
"""

# Perplexity 
print('\nPerplexity: ', lda_model.log_perplexity(corpus)) # a measure of how good the model is. lower the better.
 
# Coherence Score
coherence_model_lda = CoherenceModel(model=lda_model, texts=result_data, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nCoherence Score: ', coherence_lda)
# Perplexity는 작을 수록 Coherence Score는 높을 수록 좋다.모델 1개의 값
# 토픽의 개수를 다르게 하여 판단비교해보세요. 
# 젠심 코히러런스로 검색해봐서 coherence='c_v'값을 바꿔가면서 해보세요

#### LSA 모델

In [ ]:
from gensim import models

In [ ]:
NUM_TOPIC_WORDS = 10

In [ ]:
# ★ 모델링 후 각 토픽별로 중요한 단어들을 표시
def print_topic_words(model):
    
    for topic_id in range(model.num_topics):
        topic_word_probs = model.show_topic(topic_id, NUM_TOPIC_WORDS)
        print("Topic ID: {}".format(topic_id))

        for topic_word, prob in topic_word_probs:
            print("\t{}\t{}".format(topic_word, prob))
        print("\n")

In [ ]:
model_LSA = models.lsimodel.LsiModel(corpus, num_topics=NUM_TOPICS, id2word=corpora.Dictionary(result_data))

In [ ]:
print_topic_words(model_LSA)                  # 전체 토픽별 주요 어휘 출력 

### LDA 토픽별 키워드 조회

In [ ]:
for topic_id in range(NUM_TOPICS):
    topic_word_probs = lda_model.show_topic(topic_id, TOPICS_W_NUM)
    print("Topic ID: {}".format(topic_id))

    for topic_word, prob in topic_word_probs:
        print("\t{}\t{}".format(topic_word, prob))
    print("\n")

### 시각화

In [ ]:
"""
위의 셀에서 학습한 모델을 시각화 하여 HTML 파일로 저장하는 셀입니다. 
문서의 양이 많을 경우 오래 걸리니 참고 바랍니다. 
"""
def create_vis(model):
    pyLDAvis.enable_notebook()
    vis = pyLDAvis.gensim_models.prepare(model, corpus, id2word, sort_topics=False)
    pyLDAvis.save_html(vis, RESULT_SAVE_LDAVIS)
    return vis
#lda_model or optimal_model
create_vis(lda_model)

## 분석결과

- 1번 TOPIC : 위치, 사진, 식당, 층수, 건물 등이 관련이 높은 것을 보아 1번 TOPIC은 정릉시장맛집들에 대한 위치 정보 및 생김세에 관한 TOPIC일 가능성이 높다.  
- 2번 TOPIC : 식당, 생각, 국민대, 청년 등이 관련이 높은 것을 보아 2번 TOPIC은 국민대생들이 많이 가거나 국민대 근처에 있는 정릉시장맛집들일 가능성이 높다.  
- 3번 TOPIC : 느낌, 시간, 골목길, 영업, 입구, 저녁, 근처 등이 관련이 높은 것을 보아 3번 TOPIC은 정릉시장맛집들에 대한 영업 시간 및 위치 및 음식점 분위기 등에 관한 TOPIC일 가능성이 높다.  
- 4번 TOPIC : 메뉴, 출구, 대국, 북한, 도보, 식당, 민속, 우이신설선 등이 관련이 높은 것을 보아 4번 TOPIC은 우이신설선인 북한산보국문역 근처 맛집이거나 역과의 거리가 나와있는 TOPIC일 가능성이 높다.  
- 5번 TOPIC : 동네, 배달, 카레, 호호, 볶음밥 등이 관련이 높은 것을 보아 5번 TOPIC은 배달이 되는 정릉시장 맛집들에 대한 정보거나 카레, 볶음밥 등등을 파는 음식점이 관련이 있는 TOPIC일 가능성이 높다.  
- 6번 TOPIC : 식당, 골목, 아리랑, 백종원 등이 관련이 높은 것을 보아 6번 TOPIC은 백종원 골목 식당이라는 프로그램과 관련된 정릉시장 맛집일 가능성이 높다.